# Adaptive Lifelong IRL for Human-Robot Collaboration

HRI/ICRA-ready implementation with:
- Importance-weighted MaxEnt IRL on demo-augmented MDP
- Adaptive temporal weight decay (interaction_step-based timing)
- Empirically derived overlap threshold (no hardcoded values)
- Baseline comparison: NoReplay, UniformWeight, FixedDecay vs Proposed
- Held-out evaluation to distinguish reward generalisation from memorisation

## Cell 1: Imports and Domain Constants, State Tracker, Recipe Generator

In [1]:
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from src.environment import *

print_environment_ready()


470 raw state features


## Cell 2: Feature Engineering, IRL, Demo-Augmented MDP MaxEnt

In [ ]:
from collections import Counter, defaultdict
from dataclasses import asdict
import json
import os
import random
import textwrap

from src.baselines import *
import src.plots as plots
from src.adaptive_agent import AdaptiveHRCAgent
from src.models import Config, DEFAULT_CONFIG
from src.scenarios import (
    RECIPE_A_P1,
    RECIPE_A_P2,
    RECIPE_B,
    RECIPE_C,
    RECIPE_D,
    RECIPE_E,
    RECIPE_F,
    _accuracy,
    _run_observation,
    _run_online,
)
# Use HRC.ipynb or src.scenarios.runner for the maintained suite.

try:                import numpy as np
except Exception:   np = None

_HRC_NOTEBOOK_CTX = {}


def _bar(value, width=10):
    if value is None: return '-' * width
    value = max(0.0, min(1.0, float(value)))
    filled = int(round(value * width))
    return '█' * filled + '░' * (width - filled)


def _pct(value): 
    return 'n/a' if value is None else f'{value:.0%}'


def _clip(text, width):
    text = str(text)
    return text if len(text) <= width else text[: width - 1] + '…'


def _box(title, lines, width=84):
    print()
    print('  ╔' + '═' * width + '╗')
    for raw in [title]:
        for line in textwrap.wrap(raw, width=width - 1) or ['']:        print(f'  ║ {line:<{width - 1}}║')
    print('  ╠' + '─' * width + '╣')
    for raw in lines:
        for line in textwrap.wrap(raw, width=width - 1, subsequent_indent='  ') or ['']:    print(f'  ║ {line:<{width - 1}}║')
    print('  ╚' + '═' * width + '╝')


def _rows_to_dicts(rows):
    out = []
    for row in rows:
        out.append(asdict(row) if not isinstance(row, dict) else dict(row))
    return out


def _active_variant_lines(agent, limit=6):
    entries = sorted(agent.decay.active_entries(), key=lambda e: (-e.weight, e.recipe_id, e.pref_hash))
    lines = []
    for entry in entries[:limit]:
        latest = agent.belief.latest.get(entry.recipe_id) == entry.pref_hash
        status = 'latest' if latest else 'active'
        lines.append(f"{entry.recipe_id}/{entry.pref_hash[:12]}  [{_bar(entry.weight)}] {entry.weight:.2f}  {status}")
    if len(entries) > limit:    lines.append(f'... {len(entries) - limit} more active variant(s)')
    return lines or ['no active variants']


def _segment_dict(phase, label, mode, rows, recipe=None, post_eval=None, notes=None):
    rows = _rows_to_dicts(rows)
    online_rows = [row for row in rows if row.get('mode') == 'online']
    correct = sum(1 for row in online_rows if row.get('correct'))
    accuracy = correct / len(online_rows) if online_rows else None
    inferred = Counter(row.get('inferred_recipe') for row in rows if row.get('inferred_recipe'))
    events = Counter(row.get('event') for row in rows if row.get('event'))
    return {
        'phase': phase,
        'label': label,
        'mode': mode,
        'recipe': recipe or (inferred.most_common(1)[0][0] if inferred else None),
        'rows': rows,
        'steps': len(rows),
        'start_step': rows[0]['step'] if rows else None,
        'end_step': rows[-1]['step'] if rows else None,
        'accuracy': accuracy,
        'post_eval': post_eval,
        'delta': None if accuracy is None or post_eval is None else post_eval - accuracy,
        'locked': 'preference_shift_locked' in events,
        'events': dict(events),
        'notes': list(notes or []),
    }


def _record_segment(ctx, phase, label, mode, step_start, recipe=None, post_eval=None, notes=None):
    rows = ctx['agent'].step_log[step_start:]
    segment = _segment_dict(phase, label, mode, rows, recipe=recipe, post_eval=post_eval, notes=notes)
    ctx['segments'].append(segment)
    return segment


def _print_step_trace(segment, limit=12):
    rows = segment['rows']
    if not rows: return
    print('    step trace')
    for idx, row in enumerate(rows[:limit], 1):
        pred = row.get('predicted') or '—'
        mark = '·' if row.get('mode') == 'observe' else ('✓' if row.get('correct') else '✗')
        event = f"  [{row['event']}]" if row.get('event') else ''
        print(f"      {idx:02d}  true={_clip(row.get('actual'), 24):<24} pred={_clip(pred, 24):<24} {mark}{event}")
    if len(rows) > limit: print(f"      ... {len(rows) - limit} more step(s)")


def _print_replay_card(segment, agent):
    lines = [f"recipe={segment['recipe'] or '?'}  mode={segment['mode']}  steps={segment['steps']}  span={segment['start_step']}→{segment['end_step']}"]
    if segment['accuracy'] is not None:     lines.append(f"live accuracy      [{_bar(segment['accuracy'])}] {_pct(segment['accuracy'])}")
    else:                                   lines.append(f"observation only   {segment['steps']} action(s) buffered before retrain")
    
    if segment['post_eval'] is not None:
        if segment['delta'] is None:    lines.append(f"post-update eval   [{_bar(segment['post_eval'])}] {_pct(segment['post_eval'])}")
        else:                           lines.append(f"post-update eval   [{_bar(segment['post_eval'])}] {_pct(segment['post_eval'])}  ({segment['delta']:+.0%})")
    if segment['events']:               lines.append('events             ' + ', '.join(f"{k}×{v}" for k, v in segment['events'].items()))
    if segment['notes']:                lines.append('notes              ' + '; '.join(segment['notes']))
    lines.append('active variants')
    lines.extend('  ' + line for line in _active_variant_lines(agent))
    _box(f"{segment['phase']}  |  {segment['label']}", lines)
    if segment['mode'] == 'online' and (segment['locked'] or (segment['accuracy'] is not None and segment['accuracy'] < 0.95)): _print_step_trace(segment)


def _print_compact_ledger(segments, title):
    if not segments:
        return
    print()
    print('═' * 98)
    print(title)
    print('─' * 98)
    print(f"{'#':>2}  {'Label':<28}  {'Mode':<7}  {'Live':<16}  {'Post':<16}  {'Δ':>6}  {'Flags'}")
    print('─' * 98)
    for idx, segment in enumerate(segments, 1):
        live = f"[{_bar(segment['accuracy'], 8)}] {_pct(segment['accuracy'])}" if segment['accuracy'] is not None else f"obs {segment['steps']:>2}"
        post = f"[{_bar(segment['post_eval'], 8)}] {_pct(segment['post_eval'])}" if segment['post_eval'] is not None else '-'
        delta = '-' if segment['delta'] is None else f"{segment['delta']:+.0%}"
        flags = []
        if segment['locked']:   flags.append('lock')
        if segment['notes']:    flags.append(segment['notes'][0])
        print(f"{idx:>2}  {_clip(segment['label'], 28):<28}  {segment['mode']:<7}  {live:<16}  {post:<16}  {delta:>6}  {_clip(', '.join(flags) or '-', 18)}")


def _print_probe(title, probe):
    lines = [f"{recipe}: [{_bar(acc)}] {_pct(acc)}" for recipe, acc in sorted(probe.items())]
    _box(title, lines)


def _summarise_random_accuracy(events):
    by_recipe = defaultdict(list)
    for step, recipe, correct in events:    by_recipe[recipe].append(bool(correct))
    total = sum(len(v) for v in by_recipe.values())
    return {
        'total': total,
        'overall': (sum(sum(v) for v in by_recipe.values()) / total) if total else None,
        'per_recipe': {recipe: sum(vals) / len(vals) for recipe, vals in by_recipe.items()},
    }


def _print_random_summary(summary, observe_rows):
    lines = []
    if summary['overall'] is not None:
        lines.append(f"overall online accuracy  [{_bar(summary['overall'])}] {_pct(summary['overall'])} across {summary['total']} action predictions")
    lines.append(f'observation-mode actions during random simulation: {observe_rows}')
    for recipe, acc in sorted(summary['per_recipe'].items(), key=lambda item: (item[1], item[0]))[:6]:      lines.append(f"{recipe}: [{_bar(acc)}] {_pct(acc)}")
    _box('Random Simulation Summary', lines)


def _compact_segment(segment):
    return {key: value for key, value in segment.items() if key != 'rows'}


def _reset_notebook_ctx():
    cfg = Config(**asdict(DEFAULT_CONFIG))
    random.seed(cfg.seed)
    if np is not None:
        np.random.seed(cfg.seed)
    _HRC_NOTEBOOK_CTX.clear()
    _HRC_NOTEBOOK_CTX.update({
        'cfg': cfg,
        'agent': AdaptiveHRCAgent(cfg=cfg),
        'reports': [],
        'segments': [],
        'cf_probe_before_prune': {},
        'cf_probe_after_prune': {},
        'cf_probe_final': {},
        'random_sim': {},
        'random_sim_summary': {},
        'result': None,
        'stage': 0,
    })
    return _HRC_NOTEBOOK_CTX


def _require_ctx(phase_name):
    if 'agent' not in _HRC_NOTEBOOK_CTX:    raise RuntimeError(f'Run phase 1 before {phase_name}.')
    return _HRC_NOTEBOOK_CTX


def _build_notebook_result(ctx):
    return {
        'config': asdict(ctx['cfg']),
        'phase_reports': [asdict(report) for report in ctx['reports']],
        'cf_probe_before_prune': ctx['cf_probe_before_prune'],
        'cf_probe_after_prune': ctx['cf_probe_after_prune'],
        'cf_probe_final': ctx['cf_probe_final'],
        'random_sim': ctx['random_sim'],
        'random_sim_summary': ctx['random_sim_summary'],
        'notebook_segments': [_compact_segment(segment) for segment in ctx['segments']],
        'rate_history': ctx['agent'].decay.rate_history,
        'weight_history': [{'step': step, 'recipe': key[0], 'pref': key[1], 'w': weight}    for step, key, weight in ctx['agent'].decay.weight_history],
        'prune_events': [{'step': step, 'recipe': key[0], 'pref': key[1]}                   for step, key in ctx['agent'].decay.prune_events],
        'reentry_events': [{'step': step, 'recipe': key[0], 'pref': key[1], 'gap': gap}     for step, key, gap in ctx['agent'].decay.reentry_events],
        'retrain_events': ctx['agent'].retrain_events,
        'accuracy_events': [{'step': step, 'recipe': recipe, 'correct': correct}            for step, recipe, correct in ctx['agent'].accuracy_events],
        'step_log': [asdict(row) for row in ctx['agent'].step_log],
    }


def print_learning_ready():
    print('Learning stack ready: adaptive agent, baselines, and replay-card notebook layer')


def print_baselines_ready():
    print('Baseline systems ready: NoReplayIRL, UniformWeightIRL, FixedDecayIRL, EWCReplayIRL, NearestNeighborDemo')


def print_config_ready():
    cfg = DEFAULT_CONFIG
    print(f"Config ready: seed={cfg.seed}, max_adaptation_latency={cfg.max_adaptation_latency}, figures_dir='{cfg.figures_dir}'")


def print_helpers_ready():
    print('Notebook reporting helpers ready: replay cards, compact ledgers, selective step traces')


print_learning_ready()


Learning stack ready: adaptive agent, baselines, and replay-card notebook layer


## Preferences and Adaptive IRL

In [ ]:
# Preference system and adaptive agent now live in src/preferences.py and src/adaptive_agent.py
print_baselines_ready()

from src.scenarios import run_publication_harness
print_config_ready()

print_helpers_ready()

## Phase 1: Initial Training

In [ ]:
def run_phase_1():
    ctx = _reset_notebook_ctx()
    step_start = len(ctx['agent'].step_log)
    report = phase1_baseline(ctx['agent'])
    ctx['reports'].append(report)
    segment = _record_segment(ctx, 'Phase 1', 'Recipe A / P1 observation', 'observe', step_start, recipe='A', post_eval=report.metrics.get('accuracy'), notes=report.notes)
    _print_replay_card(segment, ctx['agent'])
    _print_compact_ledger(ctx['segments'], 'Phase 1 Ledger')
    ctx['stage'] = 1
    
run_phase_1()



=== Phase 1: Baseline learning (observation of Recipe A, P1) ===
  one-shot prediction accuracy on A/P1: 100.00%

  ╔════════════════════════════════════════════════════════════════════════════════════╗
  ║ Phase 1  |  Recipe A / P1 observation                                              ║
  ╠────────────────────────────────────────────────────────────────────────────────────╣
  ║ recipe=A  mode=observe  steps=25  span=1→25                                        ║
  ║ observation only   25 action(s) buffered before retrain                            ║
  ║ post-update eval   [██████████] 100%                                               ║
  ║ events             observing×25                                                    ║
  ║ notes              accuracy=100.00%                                                ║
  ║ active variants                                                                    ║
  ║   R1/0:transfer (  [██████████] 1.00  latest                                    

## Phase 2: Online Observation and Adaptive Retraining

In [ ]:
def run_phase_2():
    ctx = _require_ctx('phase 2')
    if ctx['stage'] >= 2:
        print('Phase 2 already executed; rerun phase 1 to reset the notebook state.')
        return
    if ctx['stage'] != 1:
        raise RuntimeError('Run phase 1 before phase 2.')

    new_segments = []

    step_start = len(ctx['agent'].step_log)
    report = phase2_same_pref(ctx['agent'])
    ctx['reports'].append(report)
    segment = _record_segment(ctx, 'Phase 2', 'Recipe A / P1 replay', 'online', step_start, recipe='A', post_eval=ctx['agent'].evaluate_sequence(RECIPE_A_P1), notes=report.notes)
    new_segments.append(segment)
    _print_replay_card(segment, ctx['agent'])

    step_start = len(ctx['agent'].step_log)
    report = phase3_pref_shift(ctx['agent'], ctx['cfg'])
    ctx['reports'].append(report)
    segment = _record_segment(ctx, 'Phase 3', 'Recipe A / P2 preference shift', 'online', step_start, recipe='A', post_eval=ctx['agent'].evaluate_sequence(RECIPE_A_P2), notes=report.notes)
    new_segments.append(segment)
    _print_replay_card(segment, ctx['agent'])

    step_start = len(ctx['agent'].step_log)
    report = phase4_new_recipe(ctx['agent'])
    ctx['reports'].append(report)
    segment = _record_segment(ctx, 'Phase 4', 'Recipe B observation', 'observe', step_start,recipe='B', post_eval=ctx['agent'].evaluate_sequence(RECIPE_B), notes=report.notes)
    new_segments.append(segment)
    _print_replay_card(segment, ctx['agent'])

    print()
    print('=== Phase 5: Rotation A/P1, A/P2, B, add C ===')
    for label, seq, rid, mode in [
        ('Rotation: A / P1', RECIPE_A_P1, 'A', 'online'),
        ('Rotation: A / P2', RECIPE_A_P2, 'A', 'online'),
        ('Rotation: B replay', RECIPE_B, 'B', 'online'),
        ('Rotation: C observation', RECIPE_C, 'C', 'observe'),
    ]:
        step_start = len(ctx['agent'].step_log)
        if mode == 'online':
            hits = _run_online(ctx['agent'], seq, rid)
            post_eval = ctx['agent'].evaluate_sequence(seq)
            notes = [f'accuracy={_accuracy(hits):.2%}']
        else:
            _run_observation(ctx['agent'], seq, recipe_hint=rid)
            post_eval = ctx['agent'].evaluate_sequence(seq)
            notes = [f'self_eval={post_eval:.2%}']
        segment = _record_segment(ctx, 'Phase 5', label, mode, step_start, recipe=rid, post_eval=post_eval, notes=notes)
        new_segments.append(segment)
        _print_replay_card(segment, ctx['agent'])

    weights = ctx['agent'].decay.weights()
    ctx['reports'].append(PhaseReport( name='phase5_rotation', passed=len(weights) >= 3, metrics={'active_variants': len(weights)}))
    ctx['cf_probe_before_prune'] = _cf_probe(ctx['agent'], {'A': RECIPE_A_P1, 'B': RECIPE_B, 'C': RECIPE_C})

    _print_compact_ledger(new_segments, 'Phase 2-5 Ledger')
    _print_probe('Forgetting Snapshot Before Prune', ctx['cf_probe_before_prune'])
    ctx['stage'] = 2

run_phase_2()



=== Phase 2: Recipe A / P1 replayed online ===
  online accuracy: 100.00%

  ╔════════════════════════════════════════════════════════════════════════════════════╗
  ║ Phase 2  |  Recipe A / P1 replay                                                   ║
  ╠────────────────────────────────────────────────────────────────────────────────────╣
  ║ recipe=A  mode=online  steps=25  span=26→50                                        ║
  ║ live accuracy      [██████████] 100%                                               ║
  ║ post-update eval   [██████████] 100%  (+0%)                                        ║
  ║ notes              accuracy=100.00%                                                ║
  ║ active variants                                                                    ║
  ║   R1/0:transfer (  [██████████] 1.00  latest                                       ║
  ╚════════════════════════════════════════════════════════════════════════════════════╝

=== Phase 3: Recipe A performed w

## Phase 3: Final Evaluation

In [ ]:
def run_phase_3():
    ctx = _require_ctx('phase 3')
    if ctx['stage'] >= 3:
        print('Phase 3 already executed; rerun phase 1 to reset the notebook state.')
        return
    if ctx['stage'] != 2:
        raise RuntimeError('Run phase 2 before phase 3.')

    new_segments = []

    report = phase6_decay_prune(ctx['agent'])
    ctx['reports'].append(report)
    _box('Phase 6  |  Decay and Pruning',[f"pruned variants  {int(report.metrics.get('pruned', 0))}",    f"active variants  {len(ctx['agent'].decay.active_entries())}",    f"global decay     {ctx['agent'].decay.global_rate:.4f}",])
    ctx['cf_probe_after_prune'] = _cf_probe(ctx['agent'], {'A': RECIPE_A_P1, 'B': RECIPE_B, 'C': RECIPE_C})

    print()
    print('=== Phase 7: Re-demo Recipe A — decay-rate adaptation ===')
    rate_before = ctx['agent'].decay.global_rate
    step_start = len(ctx['agent'].step_log)
    hits = _run_online(ctx['agent'], RECIPE_A_P1, 'A')
    rate_after = ctx['agent'].decay.global_rate
    active_a = [entry for entry in ctx['agent'].decay.active_entries() if entry.recipe_id == 'A']
    reentry_hits = [entry for entry in active_a if abs(entry.weight - 1.0) < 0.3]
    ctx['reports'].append(
        PhaseReport(
            name='phase7_reentry',
            passed=len(reentry_hits) >= 1 and rate_after != rate_before,
            metrics={'rate_before': rate_before, 'rate_after': rate_after, 'reentered_at_1': len(reentry_hits)},
        )
    )
    segment = _record_segment(ctx, 'Phase 7', 'Recipe A re-entry', 'online', step_start, recipe='A', post_eval=ctx['agent'].evaluate_sequence(RECIPE_A_P1), notes=[f'rate={rate_before:.4f}→{rate_after:.4f}', f'accuracy={_accuracy(hits):.2%}'])
    new_segments.append(segment)
    _print_replay_card(segment, ctx['agent'])

    print()
    print('=== Phase 8: Stress — add D, E, F rapidly ===')
    rate_before = ctx['agent'].decay.global_rate
    for rid, seq in [('D', RECIPE_D), ('E', RECIPE_E), ('F', RECIPE_F)]:
        step_start = len(ctx['agent'].step_log)
        _run_observation(ctx['agent'], seq, recipe_hint=rid)
        segment = _record_segment(ctx, 'Phase 8', f'Recipe {rid} observation', 'observe', step_start, recipe=rid, post_eval=ctx['agent'].evaluate_sequence(seq))
        new_segments.append(segment)
        _print_replay_card(segment, ctx['agent'])
    rate_after = ctx['agent'].decay.global_rate
    n_active = len(ctx['agent'].decay.active_entries())
    ctx['reports'].append(
        PhaseReport(
            name='phase8_stress',
            passed=rate_after <= rate_before + 1e-6 and n_active >= 4,
            metrics={'rate_before': rate_before, 'rate_after': rate_after, 'active': n_active},
        )
    )

    acc_start = len(ctx['agent'].accuracy_events)
    step_start = len(ctx['agent'].step_log)
    ctx['random_sim'] = random_simulation(ctx['agent'], ctx['cfg'])
    acc_slice = ctx['agent'].accuracy_events[acc_start:]
    step_slice = ctx['agent'].step_log[step_start:]
    ctx['random_sim_summary'] = _summarise_random_accuracy(acc_slice)

    ctx['cf_probe_final'] = _cf_probe(ctx['agent'], {'A': RECIPE_A_P1, 'B': RECIPE_B, 'C': RECIPE_C})
    ctx['result'] = _build_notebook_result(ctx)

    _print_compact_ledger(new_segments, 'Phase 7-8 Ledger')
    _print_probe('Forgetting Snapshot After Prune', ctx['cf_probe_after_prune'])
    _print_probe('Final Forgetting Snapshot', ctx['cf_probe_final'])
    _print_random_summary(ctx['random_sim_summary'], sum(1 for row in step_slice if row.mode == 'observe'))
    ctx['stage'] = 3

run_phase_3()


=== Phase 6: Idle 40 steps to force pruning ===
  pruned 5 variant(s): [('R2', '0:transfer (pot,'), ('R1', '0:transfer (pot,'), ('R1', '0:transfer (pot,'), ('R1', '0:transfer (pot,'), ('C', '0:transfer (bowl')]

  ╔════════════════════════════════════════════════════════════════════════════════════╗
  ║ Phase 6  |  Decay and Pruning                                                      ║
  ╠────────────────────────────────────────────────────────────────────────────────────╣
  ║ pruned variants  5                                                                 ║
  ║ active variants  0                                                                 ║
  ║ global decay     0.2236                                                            ║
  ╚════════════════════════════════════════════════════════════════════════════════════╝

=== Phase 7: Re-demo Recipe A — decay-rate adaptation ===

  ╔════════════════════════════════════════════════════════════════════════════════════╗
  ║ Phase 7  | 

## Phase 4: Structured Results (Tables 0–6)

In [ ]:
def run_phase_4():
    ctx = _require_ctx('phase 4')
    if ctx['stage'] < 3:
        raise RuntimeError('Run phase 3 before phase 4.')
    ctx['result'] = _build_notebook_result(ctx)
    out_dir = ctx['cfg'].figures_dir
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, 'harness_log.json')
    with open(out_path, 'w') as fh:
        json.dump(ctx['result'], fh, indent=2, default=str)

    _print_compact_ledger(ctx['segments'], 'All Notebook Segments')
    _box(
        'Structured Results',
        [
            f"reports written        {len(ctx['reports'])}",
            f"segment summaries      {len(ctx['segments'])}",
            f"before prune probe     {ctx['cf_probe_before_prune']}",
            f"after prune probe      {ctx['cf_probe_after_prune']}",
            f"final probe            {ctx['cf_probe_final']}",
            f"random simulation      {ctx['random_sim']}",
            f"json artifact          {out_path}",
        ],
    )
    ctx['stage'] = 4
    
run_phase_4()


══════════════════════════════════════════════════════════════════════════════════════════════════
All Notebook Segments
──────────────────────────────────────────────────────────────────────────────────────────────────
 #  Label                         Mode     Live              Post                   Δ  Flags
──────────────────────────────────────────────────────────────────────────────────────────────────
 1  Recipe A / P1 observation     observe  obs 25            [████████] 100%        -  accuracy=100.00%
 2  Recipe A / P1 replay          online   [████████] 100%   [████████] 100%      +0%  accuracy=100.00%
 3  Recipe A / P2 preference sh…  online   [███████░] 88%    [████████] 100%     +12%  lock, accuracy=88…
 4  Recipe B observation          observe  obs 23            [████████] 96%         -  known=['R1', 'R2']
 5  Rotation: A / P1              online   [████████] 96%    [████████] 100%      +4%  lock, accuracy=96…
 6  Rotation: A / P2              online   [████████] 96%    

## Visualisation

In [ ]:
def run_visualisation():
    ctx = _require_ctx('visualisation')
    if ctx['stage'] < 3:
        raise RuntimeError('Run phase 3 before visualisation.')
    if ctx['result'] is None:
        ctx['result'] = _build_notebook_result(ctx)
    plots.render_all(ctx['result'], out_dir=ctx['cfg'].figures_dir, cfg=ctx['cfg'])
    _box(
        'Visualisation Artifacts',
        [
            f"output directory       {ctx['cfg'].figures_dir}",
            'accuracy_over_time.png',
            'decay_weights.png',
            'global_decay_rate.png',
            'next_action_confusion.png',
            'catastrophic_forgetting.png',
        ],
    )


run_visualisation()

/home/abd/Documents/project/src/plots.py:125: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()



  ╔════════════════════════════════════════════════════════════════════════════════════╗
  ║ Visualisation Artifacts                                                            ║
  ╠────────────────────────────────────────────────────────────────────────────────────╣
  ║ output directory       paper_figures                                               ║
  ║ accuracy_over_time.png                                                             ║
  ║ decay_weights.png                                                                  ║
  ║ global_decay_rate.png                                                              ║
  ║ next_action_confusion.png                                                          ║
  ║ catastrophic_forgetting.png                                                        ║
  ╚════════════════════════════════════════════════════════════════════════════════════╝


## Cell 2.0 — N-Gram Markov, State Transition Graph, and IRL+Markov Ensemble

The proposed system couples the IRL reward predictor with a weighted-replay n-gram Markov predictor. Intuition: IRL captures goal-directed preferences but generalises poorly from a single demo; the Markov predictor captures local action-sequence regularities and can be queried fast. The ensemble takes a convex combination with a confidence-adaptive mixing weight.


In [12]:
# NGramMarkov, StateTransitionGraph, and the ensemble predictor now live in src/irl.py


## Cell 2.1 — AdaptiveIRL v2 (ensemble of IRL + n-gram Markov)

`AdaptiveIRLv2` subclasses `AdaptiveIRL` and overrides `_retrain` to (a) also fit the n-gram Markov on the weighted buffer and (b) build the state transition graph.  `predict_for_demo` exposes the ensemble path. The ablations `IRLOnly` and `MarkovOnly` isolate each head.


In [13]:
# Adaptive agent and evaluation helpers now live in src/adaptive_agent.py and src/experiments.py


## Cell 2.2 — Extended baselines for the 2.0 benchmark

- **IRLOnly** — AdaptiveIRLv2 with `use_ensemble=False` (ablates the Markov head).
- **MarkovOnly** — no IRL; only n-gram Markov fit on the weighted replay buffer.
- **NearestNeighborDemo** — predict by copying the action of the nearest training state (state-feature cosine similarity), no learning at all.
- **EWCReplayIRL** — L2-regularised IRL against the previous cycle's weights (Elastic Weight Consolidation approximation); tests whether EWC-style consolidation alone delivers the same forgetting resistance without adaptive decay.


In [14]:
# Extended baselines now live in src/baselines.py


## Cell 2.3 — Extended online test sequence (100 cases, procedural)

50 structured cases (Groups A–H) exercising every cell of the 2×2 novelty/preference table on every recipe family. 50 random cases (drawn from the recipe pool × preference pool with an occasional repeat) simulate a long realistic interaction run so decay/reentry/decay-rate updates are visible. Categories are derived from comparison with the *initial* training demo so that new recipes or unseen preference profiles are consistently tagged.

Importantly: this procedural generator does **not** tell the system which recipe is which — the label we pass is treated by `AdaptiveIRL` as an opaque bookkeeping string. Novelty is inferred from the action sequence alone.


In [15]:
# Extended test sequence generation now lives in src/experiments.py


## Cell 2.4 — Seed loop + publication-grade metric suite

For each seed:
1. Seed a system with one initial demo (same recipe, same preference profile as Phase 1).
2. Run the 100-case extended test sequence session-by-session (session size = 5).
3. At every session end, retrain and snapshot: per-category accuracy, novelty-classification precision/recall/F1, active-set size, and BWT (accuracy on every previously ingested demo against the *current* model).
4. After the final session, record the end-to-end summary.

Aggregate mean ± std across seeds for every metric. Metrics:
- **Per-category accuracy** (same/same, same/diff, diff/same, diff/diff) — how the system handles each novelty quadrant.
- **Novelty F1** — classification accuracy with respect to the human-flag oracle.
- **Backward Transfer (BWT)** — mean over previously-seen demos of (acc_current − acc_at_ingestion); negative BWT signals forgetting.
- **Adaptation speed** — for each novel case, how many cycles until that demo's accuracy crosses 80%.
- **Memory footprint** — mean active-demo count across the run.


In [16]:
# Seed-loop and aggregate benchmark helpers now live in src/experiments.py


## Cell 2.5 — v2 aggregate tables and publication plots

Runs the multi-seed loop, prints the paper tables with mean ± std, and produces a 6-panel publication figure (per-category accuracy, novelty F1, BWT, memory footprint, forgetting, adaptive decay trajectory).


In [17]:
# Optional benchmark run:
# AGG_V2, ALL_RUNS_V2 = run_v2_benchmark()
